# M5 Forecasting — Notebook 2: Feature Engineering

**Pipeline:** EDA → **Feature Engineering** → Forecasting → Optimization

Processes one store at a time — peak RAM ~400 MB per store.
Each store's features are written to a parquet file immediately and
never concatenated, so the 57M-row dataset never lives in RAM at once.

> Set `N_ITEMS = 500` for a fast dev run, `None` for the full dataset.


In [ ]:
import sys, pathlib, os

# ── Source files path ──────────────────────────────────────────────────────────
# Kaggle  : set to your dataset path
# Local   : set to the project root (folder containing src/)
# Override: export SRC_ROOT=/your/path before running
SRC_ROOT = os.environ.get(
    "SRC_ROOT",
    "/path/to/the/folder/containing/src/and/config.py"
)

if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

# ── Load config (handles all other paths automatically) ────────────────────────
from config import print_config, DATA_DIR, CACHE_DIR, OUTPUT_DIR, MODEL_DIR, MLFLOW_URI
print_config()


In [ ]:
import gc
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s | %(levelname)s | %(message)s',
                    datefmt='%H:%M:%S')

from src.data.features import (
    build_features, FEATURE_COLS, TARGET_COL,
)

N_ITEMS   = None    # set to 500 for fast dev run, None for full dataset
CACHE_DIR = CACHE_DIR

COLORS = {'green':'#48bb78','blue':'#63b3ed','orange':'#f6ad55',
          'red':'#fc8181','gray':'#718096','purple':'#b794f4'}
plt.rcParams.update({
    'figure.facecolor':'#0f1117','axes.facecolor':'#1a202c',
    'axes.edgecolor':'#2d3748','axes.labelcolor':'#e2e8f0',
    'xtick.color':'#a0aec0','ytick.color':'#a0aec0',
    'text.color':'#e2e8f0','grid.color':'#2d3748',
})
print('Imports OK ✓')


## Build Features (per-store, resumable)

In [ ]:
import shutil
from pathlib import Path

# Remove old cache so any changes to features.py take effect
cache_path = Path(CACHE_DIR)
if cache_path.exists():
    shutil.rmtree(cache_path)
    print('Old cache cleared')

# Processes one store at a time and writes features_<store>.parquet
# Returns the cache folder Path — never loads all stores into RAM at once
cache_dir = build_features(
    DATA_DIR,
    n_items   = N_ITEMS,
    cache_dir = CACHE_DIR,
)

files = sorted(cache_dir.glob('features_*.parquet'))
print(f'\nDone. {len(files)} store parquets written to {cache_dir}')
for f in files:
    print(f'  {f.name} — {f.stat().st_size / 1e6:.0f} MB')


## Summary

In [ ]:
# Read one store to verify columns, shape, NaNs
sample = pd.read_parquet(files[0])

print(f'Rows per store  : {len(sample):,}')
print(f'Total rows      : ~{len(sample) * len(files):,}  ({len(files)} stores)')
print(f'Columns         : {sample.shape[1]}')
print(f'Feature cols    : {len(FEATURE_COLS)}')
print(f'NaNs in X       : {sample[FEATURE_COLS].isna().sum().sum()}')
print(f'Date range      : {sample["date"].min().date()} -> {sample["date"].max().date()}')
print(f'Zero-sales rate : {(sample[TARGET_COL] == 0).mean():.1%}')
del sample; gc.collect()


## Feature Distributions (sample from one store)

In [ ]:
sample = pd.read_parquet(files[0])

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Key Feature Distributions (CA_1)', fontsize=13)

pairs = [
    ('lag_7',         COLORS['blue'],   'Lag-7 Sales'),
    ('rmean_28',      COLORS['green'],  'Rolling Mean-28'),
    ('sell_price',    COLORS['orange'], 'Sell Price'),
    ('price_pct_chg', COLORS['purple'], 'Price % Change'),
    ('lag_28',        COLORS['blue'],   'Lag-28 Sales'),
    ('rstd_7',        COLORS['red'],    'Rolling Std-7'),
]
for ax, (col, color, title) in zip(axes.flatten(), pairs):
    data = sample[col].dropna()
    data = data[data.between(data.quantile(0.01), data.quantile(0.99))]
    ax.hist(data, bins=60, color=color, edgecolor='none', alpha=0.85)
    ax.set_title(title); ax.set_xlabel('Value'); ax.set_ylabel('Count')

plt.tight_layout(); plt.show()
del sample; gc.collect()


## Rolling Mean vs Actual — Sample Series

In [ ]:
sample = pd.read_parquet(files[0])
sample_id     = sample['id'].iloc[0]
sample_series = sample[sample['id'] == sample_id].sort_values('date').tail(200)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(sample_series['date'], sample_series['sales'],
        color=COLORS['gray'], alpha=0.6, lw=1, label='Actual')
ax.plot(sample_series['date'], sample_series['rmean_7'],
        color=COLORS['green'], lw=1.5, label='rmean_7')
ax.plot(sample_series['date'], sample_series['rmean_28'],
        color=COLORS['orange'], lw=1.5, label='rmean_28')
ax.set_title(f'Rolling Features — {sample_id}')
ax.set_xlabel('Date'); ax.set_ylabel('Units'); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()
del sample; gc.collect()


## Target Distribution

In [ ]:
sample = pd.read_parquet(files[0])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(sample[TARGET_COL], bins=50,
             color=COLORS['blue'], edgecolor='none', alpha=0.85)
axes[0].set_title('Sales Distribution'); axes[0].set_xlabel('Units Sold')

nonzero = sample[sample[TARGET_COL] > 0][TARGET_COL]
axes[1].hist(nonzero, bins=50, color=COLORS['green'],
             edgecolor='none', alpha=0.85, log=True)
axes[1].set_title('Non-Zero Sales (log scale)'); axes[1].set_xlabel('Units Sold')

zero_rate = (sample[TARGET_COL] == 0).mean()
plt.suptitle(f'Target: {TARGET_COL}  |  Zero rate: {zero_rate:.1%}', fontsize=12)
plt.tight_layout(); plt.show()
del sample; gc.collect()


In [ ]:
print('Feature engineering complete.')
print(f'Per-store parquets in: {cache_dir}')
print()
print('Next -> 03_forecasting.ipynb')
